# 09 抓取放置最小闭环

## 学习目标

- 按可观察步骤完成抓取/放置链路。
- 理解 HEAD 视角、ARM 视角、bbox 匹配的关系。
- 在真实执行前检查每一步中间结果。
- 知道每个开关应该按什么顺序打开。

参考文献：文档第 4.1、6.4.5 至 6.4.11 章。

In [ ]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from helpers import DEFAULT_WS_URL

WS_URL = DEFAULT_WS_URL
print("WS_URL =", WS_URL)

In [ ]:
CONNECT_ROBOT = False
CAPTURE_HEAD = False
DETECT_HEAD = False
ENABLE_LOOKTO = False
CAPTURE_ARM = False
DETECT_ARM = False
ENABLE_PICK = False
ENABLE_PLACE = False

LABELS = ["玩具"]
PLACE_BBOX = [153.0, 230.0, 403.0, 384.0]

# 推荐开关顺序：
# 1. CONNECT_ROBOT + CAPTURE_HEAD
# 2. DETECT_HEAD
# 3. ENABLE_LOOKTO + CAPTURE_ARM
# 4. DETECT_ARM
# 5. 确认两个 bbox 都合理后 ENABLE_PICK
# 6. 确认放置框合理后 ENABLE_PLACE

In [ ]:
from helpers import (
    connect_robot, disconnect_safely, draw_bboxes, first_or_none,
    print_mission_status, require_enabled, show_rgb,
    summarize_detections, summarize_view,
)
from bajie_sdk import CameraType, DetectObjectsRequest, OvdEndpoint

robot = connect_robot(WS_URL) if CONNECT_ROBOT else None
head_view = None
head_items = []
target = None
arm_view = None
arm_items = []
matched = None

In [ ]:
if robot and CAPTURE_HEAD:
    head_view = robot.eco_captureImages(CameraType.HEAD, timeout_sec=30.0)
    show_rgb(head_view, "HEAD view")
    summarize_view(head_view)
else:
    print("Head capture disabled.")

In [ ]:
if robot and head_view is not None and DETECT_HEAD:
    resp = robot.eco_detect_objects(
        DetectObjectsRequest(
            rgb_image=head_view.rgb_image.img,
            labels=LABELS,
            entry=OvdEndpoint.OVD,
        ),
        timeout=15.0,
    )
    head_items = list(resp.items)
    summarize_detections(head_items)
    draw_bboxes(head_view, head_items, "HEAD detections")
    first = first_or_none(head_items)
    target = first.to_named_bbox_for_grab() if first else None
    print("target:", target)
else:
    print("Head detection disabled or no head_view.")

In [ ]:
if robot and target is not None and ENABLE_LOOKTO:
    st = robot.eco_lookto(
        target.bbox,
        view=head_view,
        camera_type=CameraType.ARM,
        timeout_sec=30.0,
    )
    print_mission_status(st)
else:
    print("lookto disabled or no target.")

In [ ]:
if robot and CAPTURE_ARM:
    arm_view = robot.eco_captureImages(CameraType.ARM, timeout_sec=30.0)
    show_rgb(arm_view, "ARM view")
    summarize_view(arm_view)
else:
    print("Arm capture disabled.")

In [ ]:
if robot and arm_view is not None and DETECT_ARM:
    label = (getattr(target, "name", "") or "object") if target else "object"
    resp = robot.eco_detect_objects(
        DetectObjectsRequest(
            rgb_image=arm_view.rgb_image.img,
            labels=[label],
            entry=OvdEndpoint.OVD,
        ),
        timeout=15.0,
    )
    arm_items = list(resp.items)
    summarize_detections(arm_items)
    draw_bboxes(arm_view, arm_items, "ARM detections")
    if target and arm_items:
        matched = robot.eco_match_obj_views(head_view.rgb_image.img, target.bbox, arm_view.rgb_image.img, arm_items)
        print("matched:", matched)
else:
    print("Arm detection disabled or no arm_view.")

In [ ]:
if robot and matched is not None and ENABLE_PICK:
    require_enabled(ENABLE_PICK, "eco_pick")
    st = robot.eco_pick(arm_view, matched, timeout_sec=120.0)
    print_mission_status(st)
else:
    print("Pick disabled or no matched bbox.")

In [ ]:
if robot and head_view is not None and ENABLE_PLACE:
    require_enabled(ENABLE_PLACE, "eco_place_in")
    st = robot.eco_place_in(head_view, PLACE_BBOX, timeout_sec=120.0)
    print_mission_status(st)
else:
    print("Place disabled or no head_view.")

In [ ]:
disconnect_safely(robot)

## 机器人背景知识

- 抓取前要确认目标框准不准。
- HEAD 视角和 ARM 视角不是同一张图，所以需要 `eco_match_obj_views`。
- 放置可以使用像素框、像素点或 3D pose，最小入门先使用容器框。

## 故障排查卡片

- bbox 不准：重新拍照、换标签、换角度。
- ARM 视角没看到物体：重新 `lookto` 或调整检测标签。
- 抓取失败：先恢复姿态，确认环境安全，再继续。

## 小练习

在真正打开 `ENABLE_PICK` 前，截图或记录 HEAD/ARM 两个检测框是否都合理。